# Reencuadre fenológico — ablación completa de bloques opcionales

Cuantifica el aporte de los **bloques opcionales** sobre el conjunto fused completo:

1. **FarSLIP** (embeddings de 512 dimensiones, extracción real del epoch 2).
2. **Descripción fenológica textual con Gemini 3.5 Flash** codificada con sentence-transformers.
3. **Firma espectral REP** (Frampton et al. 2013) calculada desde anclas Sentinel-2 muestreadas en Earth Engine.

**Comportamiento ante datos faltantes**: si `GEMINI_API_KEY` no está definida o `PASTIS-R/` no está en disco, el cuaderno lanza error explícito con instrucciones, sin saltarse pasos en silencio. Si los parquets de los bloques no existen, los materializa directamente desde aquí.

La ablación reproduce los conjuntos de `04c_baseline.ipynb` y añade:

- `with_farslip` / `farslip_only`
- `with_pheno_text` / `pheno_text_only`
- `with_spectral_signature` / `spectral_signature_only`

In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FUSED_PATH = "data/features/features_fused_italy.parquet"
PHENO_TEXT_PATH = "data/features/phenology_text_italy.parquet"
S2_ANCHORS_PATH = "data/features/s2_anchors_italy.parquet"
SPECTRAL_SIGNATURE_PATH = "data/features/spectral_signature_italy.parquet"
FARSLIP_PATH = "data/farslip/embeddings_italy.parquet"
FIGURES_SUBDIR = "us-023-preview/05_reencuadre"
REPORTS_SUBDIR = "baseline/05_reencuadre"
YEAR = 2023
K_FOLDS = 5
BUFFER_KM = 1.0
MAX_SAMPLES = None
ENABLE_FARSLIP = True
ENABLE_PHENO_TEXT = True
ENABLE_SPECTRAL_SIGNATURE = True
ENFORCE_GEMINI_API_KEY = True


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Carga del dataset base (con metadata)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    materialize_phenology_text_if_missing,
    materialize_s2_anchors_if_missing,
    materialize_spectral_signature_if_missing,
    run_ablation_and_persist,
)
from ml.utils.parcel_id import canonical_parcel_id
from ml.eval.reencuadre_plots import (
    plot_ablation_bars,
    plot_geom_leakage_comparison,
    plot_optional_blocks_ablation,
)
from ml.eval.feature_ablation import FeatureAblationResult
from pathlib import Path

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
display(Markdown(f'**Dataset base**: `{df.height:,}` parcelas x `{df.width}` cols'))


## Materialización del bloque `pheno_text` (Gemini sobre el dataset completo)

In [ ]:
if ENABLE_PHENO_TEXT:
    if ENFORCE_GEMINI_API_KEY and not env.has_gemini_api_key:
        raise RuntimeError(
            'GEMINI_API_KEY ausente. Define la variable en `.env.local` antes de re-ejecutar, '
            'o pon ENFORCE_GEMINI_API_KEY=False para correr solo las ablaciones base.'
        )
    pheno_path = materialize_phenology_text_if_missing(
        parcels_features_path=FEATURES_PATH,
        output_path=PHENO_TEXT_PATH,
        enforce_api_key=ENFORCE_GEMINI_API_KEY,
    )
    pheno_df = canonical_parcel_id(pl.read_parquet(pheno_path))
    display(Markdown(f'**pheno_text**: `{pheno_df.shape}` en `{pheno_path}`'))
else:
    pheno_df = None
    display(Markdown('> ENABLE_PHENO_TEXT=False: bloque omitido.'))


## Materialización de anclas Sentinel-2 y firma espectral REP (Frampton 2013)

In [ ]:
if ENABLE_SPECTRAL_SIGNATURE:
    if not env.has_ee_credentials:
        display(Markdown(
            '> Earth Engine no configurado. Define `GEE_PROJECT_ID` '
            'en `.env.local` o ejecuta `earthengine authenticate`. '
            'El muestreo S2 anchors fallara sin esto.'
        ))
    anchors_path = materialize_s2_anchors_if_missing(
        parcels_geoparquet=PARCELS_GEOPARQUET,
        output_path=S2_ANCHORS_PATH,
        year=YEAR,
    )
    spec_path = materialize_spectral_signature_if_missing(
        s2_anchors_path=anchors_path,
        output_path=SPECTRAL_SIGNATURE_PATH,
        descriptor='rep',
    )
    spec_df = canonical_parcel_id(pl.read_parquet(spec_path))
    display(Markdown(f'**spectral_signature**: `{spec_df.shape}` en `{spec_path}`'))
else:
    spec_df = None
    display(Markdown('> ENABLE_SPECTRAL_SIGNATURE=False: bloque omitido.'))


## Carga de FarSLIP desde la ruta canónica (`parcel_id` en Utf8)

In [ ]:
if ENABLE_FARSLIP:
    farslip_path = Path(FARSLIP_PATH)
    if not farslip_path.exists():
        raise FileNotFoundError(
            f'FarSLIP parquet no encontrado en {farslip_path}. '
            'Ejecuta `dvc pull data/farslip/embeddings_italy.parquet.dvc` '
            'antes de re-ejecutar.'
        )
    farslip_df = canonical_parcel_id(pl.read_parquet(farslip_path))
    display(Markdown(f'**FarSLIP**: `{farslip_df.shape}` en `{farslip_path}` con parcel_id Utf8.'))
else:
    farslip_df = None
    display(Markdown('> ENABLE_FARSLIP=False: bloque omitido.'))


## Fusión de bloques: base + FarSLIP + pheno_text + spectral_signature

Aplicamos un LEFT JOIN secuencial sobre `parcel_id` (todos en Utf8 tras `canonical_parcel_id`). Las parcelas sin coincidencia en algún bloque opcional quedan con NaN; XGBoost y LightGBM los toleran nativamente y RandomForest los imputa por mediana.

In [ ]:
df = canonical_parcel_id(df)
fused = df
joined_log = []
if farslip_df is not None:
    keep = ['parcel_id'] + [c for c in farslip_df.columns if c.startswith('farslip_')]
    fused = fused.join(farslip_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'FarSLIP: +{len(keep)-1} cols')
if pheno_df is not None:
    keep = ['parcel_id'] + [c for c in pheno_df.columns if c.startswith('pheno_text_')]
    fused = fused.join(pheno_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'pheno_text: +{len(keep)-1} cols')
if spec_df is not None:
    keep = ['parcel_id'] + [c for c in spec_df.columns if c.startswith('spectral_signature_')]
    fused = fused.join(spec_df.select(keep), on='parcel_id', how='left')
    joined_log.append(f'spectral_signature: +{len(keep)-1} cols')

display(Markdown(
    f"**Conjunto fused final**: `{fused.shape}`\n\n"
    + "\n".join(f"- {l}" for l in joined_log)
))


## Ablación con todos los bloques opcionales

In [ ]:
ablation_table, parquet_path = run_ablation_and_persist(
    fused,
    output_dir=env.reports_dir,
    models=('xgb',),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    max_samples=MAX_SAMPLES,
)
display(Markdown(f'**Tabla de ablación**: `{parquet_path.relative_to(env.repo)}`'))
display(ablation_table)


## Gráficos: ablación completa, fuga geométrica y aporte de bloques opcionales

In [ ]:
results = [
    FeatureAblationResult(
        feature_set=row['feature_set'],
        model_kind=row['model'],
        f1_macro=row['f1_macro'] if row['f1_macro'] is not None else float('nan'),
        f1_weighted=row['f1_weighted'] if row['f1_weighted'] is not None else float('nan'),
        miou=row['miou'] if row['miou'] is not None else float('nan'),
        n_features=row['n_features'],
        delta_vs_full=row['delta_vs_full'] if row['delta_vs_full'] is not None else float('nan'),
    )
    for row in ablation_table.iter_rows(named=True)
]

fig_abl = plot_ablation_bars(results, title='F1-macro por conjunto (ablación completa)')
fig_abl.savefig(env.figures_dir / 'ablation_full.png', bbox_inches='tight')
display(fig_abl)
plt.close(fig_abl)

fig_geom = plot_geom_leakage_comparison(results)
fig_geom.savefig(env.figures_dir / 'geom_leakage.png', bbox_inches='tight')
display(fig_geom)
plt.close(fig_geom)

fig_opt = plot_optional_blocks_ablation(results)
fig_opt.savefig(env.figures_dir / 'optional_blocks.png', bbox_inches='tight')
display(fig_opt)
plt.close(fig_opt)


## Conclusiones — decisión por bloque

Las decisiones (promover, descartar o diferir) se toman bloque por bloque siguiendo el umbral de mejora `delta >= +0.005`:

1. **FarSLIP**: si `with_farslip - full >= +0.005`, FarSLIP se promueve al baseline y entra al conjunto ganador. Si el delta cae en [-0.005, +0.005], se mantiene como modelo base del ensamble por apilamiento posterior. Si es menor que -0.005, se descarta del baseline.

2. **pheno_text (Gemini Flash sobre el dataset completo)**: misma regla. La ablación aquí cuantifica el aporte real de la rama semántica propuesta por Wen et al. (2025) sobre el conjunto de Italia.

3. **Firma espectral REP**: misma regla. Es la primera aplicación del descriptor Frampton 2013 sobre el dataset Italia.

4. **`geom_only`**: si F1-macro < 0.10, se confirma que no hay leakage espacial agronómicamente significativo y la decisión previa de descartar `geom_*` queda validada con evidencia cuantitativa.

## Lo que sigue

- `Avance3.Equipo17.ipynb` lee esta `ablation_table.parquet`, ejecuta `select_winning_features()` y persiste `features_fused_winning_italy.parquet` que consumen los modelos densos siguientes.